In [1]:
# 1. Instalar Java (Prerrequisito de Hadoop)
!apt-get install openjdk-8-jdk-headless -qq > /dev/null

# 2. Descargar Apache Hadoop Oficial
print("Descargando Hadoop (esto puede tomar 10-15 segundos)...")
!wget -q https://archive.apache.org/dist/hadoop/common/hadoop-3.3.6/hadoop-3.3.6.tar.gz

# 3. Descomprimir el archivo tar.gz
print("Descomprimiendo archivos...")
!tar -xzf hadoop-3.3.6.tar.gz

# 4. Configurar variables de entorno indispensables
import os
os.environ["JAVA_HOME"] = "/usr/lib/jvm/java-8-openjdk-amd64"
os.environ["HADOOP_HOME"] = "/content/hadoop-3.3.6"
os.environ["PATH"] += os.pathsep + "/content/hadoop-3.3.6/bin"

print("¡Entorno configurado correctamente!")

Descargando Hadoop (esto puede tomar 10-15 segundos)...
Descomprimiendo archivos...
¡Entorno configurado correctamente!


In [2]:
%%writefile mapper.py
#!/usr/bin/env python3
import sys
import re

def main():
    # El Mapper recibe datos desde la entrada estándar (stdin)
    for linea in sys.stdin:
        # Limpiar espacios en blanco laterales y transformar a minúsculas
        linea = linea.strip().lower()
        # Eliminar signos de puntuación mediante expresiones regulares
        linea = re.sub(r'[^\w\s]', '', linea)
        # Separar el texto por palabras
        palabras = linea.split()

        # Emitir cada palabra encontrada estructurada como clave-valor
        for palabra in palabras:
            if palabra: # Evitar procesar strings vacíos
                print(f"{palabra}\t1")

if __name__ == "__main__":
    main()

Writing mapper.py


In [3]:
%%writefile reducer.py
#!/usr/bin/env python3
import sys

def main():
    palabra_actual = None
    conteo_actual = 0

    # Leer el flujo ordenado enviado desde stdin
    for linea in sys.stdin:
        linea = linea.strip()
        try:
            palabra, conteo = linea.split('\t', 1)
            conteo = int(conteo)
        except ValueError:
            continue # Descartar líneas mal formateadas

        # Lógica de acumulación basada en el ordenamiento de claves
        if palabra_actual == palabra:
            conteo_actual += conteo
        else:
            if palabra_actual:
                # Emitir resultado consolidado al cambiar de palabra
                print(f"{palabra_actual}\t{conteo_actual}")
            palabra_actual = palabra
            conteo_actual = conteo

    # Imprimir la última palabra procesada por el stream
    if palabra_actual == palabra:
        print(f"{palabra_actual}\t{conteo_actual}")

if __name__ == "__main__":
    main()

Writing reducer.py


In [9]:
%%writefile mi_dataset.txt
Hadoop es un framework de Big Data.
Big Data requiere procesamiento distribuido y MapReduce ayuda en esto.
El modelo MapReduce procesa datos masivos en paralelo con un mapper y un reducer.
Hadoop y MapReduce son herramientas fundamentales de Big Data.

Overwriting mi_dataset.txt


In [12]:
%%bash
chmod +x mapper.py reducer.py
STREAMING_JAR=$HADOOP_HOME/share/hadoop/tools/lib/hadoop-streaming-3.3.6.jar

# Borramos la carpeta por si ya existe
rm -rf resultado_wordcount_1

$HADOOP_HOME/bin/hadoop jar $STREAMING_JAR \
  -files mapper.py,reducer.py \
  -mapper "python3 mapper.py" \
  -reducer "python3 reducer.py" \
  -numReduceTasks 1 \
  -input mi_dataset.txt \
  -output resultado_wordcount_1

2026-07-08 22:42:52,192 INFO impl.MetricsConfig: Loaded properties from hadoop-metrics2.properties
2026-07-08 22:42:52,295 INFO impl.MetricsSystemImpl: Scheduled Metric snapshot period at 10 second(s).
2026-07-08 22:42:52,296 INFO impl.MetricsSystemImpl: JobTracker metrics system started
2026-07-08 22:42:52,318 WARN impl.MetricsSystemImpl: JobTracker metrics system already initialized!
2026-07-08 22:42:52,708 INFO mapred.FileInputFormat: Total input files to process : 1
2026-07-08 22:42:52,734 INFO mapreduce.JobSubmitter: number of splits:1
2026-07-08 22:42:52,938 INFO mapreduce.JobSubmitter: Submitting tokens for job: job_local1709836209_0001
2026-07-08 22:42:52,938 INFO mapreduce.JobSubmitter: Executing with tokens: []
2026-07-08 22:42:53,332 INFO mapred.LocalDistributedCacheManager: Localized file:/content/mapper.py as file:/tmp/hadoop-root/mapred/local/job_local1709836209_0001_5d7056bf-9680-4d86-916e-a0a4dd66df05/mapper.py
2026-07-08 22:42:53,360 INFO mapred.LocalDistributedCacheMa

In [8]:
# Listar los archivos generados dentro del directorio de salida
print("Archivos generados:")
!ls -l resultado_wordcount

print("\n--- CONTENIDO DEL RESULTADO FINAL ---")
# Mostrar el conteo exacto de palabras obtenido
!cat resultado_wordcount/part-00000

Archivos generados:
total 4
-rw-r--r-- 1 root root 241 Jul  8 22:15 part-00000
-rw-r--r-- 1 root root   0 Jul  8 22:15 _SUCCESS

--- CONTENIDO DEL RESULTADO FINAL ---
ayuda	1
big	3
con	1
data	3
datos	1
de	2
distribuido	1
el	1
en	2
es	1
esto	1
framework	1
fundamentales	1
hadoop	2
herramientas	1
mapper	1
mapreduce	3
masivos	1
modelo	1
paralelo	1
procesa	1
procesamiento	1
reducer	1
requiere	1
son	1
un	3
y	3
